# Hindi UD → 13 target languages: two simultaneous Kaggle T4 workers

**Experimental bilingual parser**, not ordinary Trankit. Requires a Kaggle notebook with **GPU T4×2** and your own Hindi gold CoNLL-U + parallel target JSONL + scored/hard alignment JSONL. Instructions: `docs/KAGGLE_DUAL_T4.md`.

Each GPU trains one language. The XLM-R backbone is frozen; source/target embeddings are cached. Only parallel graph-attention layers and target dependency heads train. **No GPU timings or target-language LAS claims have been validated yet.**


In [ ]:
import os, subprocess
from pathlib import Path
import torch
print('Visible CUDA devices:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i), 'VRAM GB:', round(torch.cuda.get_device_properties(i).total_memory/1024**3, 1))
if torch.cuda.device_count() < 2:
    raise RuntimeError('Select GPU T4x2 in the Kaggle notebook settings before running the dual launcher.')


## Step 1 — Install project

Upload `ud-treebank-lab-v0.2.0.zip` to a **private Kaggle dataset** attached as notebook input, or provide an equivalent source checkout. Internet access or a local `xlm-roberta-base` model snapshot is needed for the encoder cache stage.


In [ ]:
import zipfile
project_zip = next(Path('/kaggle/input').rglob('ud-treebank-lab-v0.2.0.zip'), None)
if project_zip is None:
    raise FileNotFoundError('Attach a Kaggle Dataset containing ud-treebank-lab-v0.2.0.zip')
with zipfile.ZipFile(project_zip) as z:
    z.extractall('/kaggle/working')
project = Path('/kaggle/working/ud-treebank-lab')
os.chdir(project)
subprocess.run(['python','-m','pip','install','-e','.[bilingual]'], check=True)
subprocess.run(['python','-m','ud_treebank_lab','--version'], check=True)


## Step 2 — Supply the corpus and manifest

Attach a private dataset with `hindi_train.conllu`, `languages.jsonl`, and target/alignment JSONL files. Every target `id` must match the source Hindi `sent_id`. The manifest lists 13 languages (`language`, `targets`, `alignments`) and absolute `/kaggle/input/...` paths. Do not include Hindi held-out test annotations in model training.


In [ ]:
DATA_ROOT = Path('/kaggle/input/my-ud-data')   # <-- change to your dataset slug
hindi_source = DATA_ROOT / 'hindi_train.conllu'
manifest = DATA_ROOT / 'languages.jsonl'
for p in (hindi_source, manifest):
    if not p.exists():
        raise FileNotFoundError(f'Change DATA_ROOT to your dataset: {p}')
CONFIG_DIR = project / 'configs' / 'generated'
CACHE_DIR = Path('/kaggle/working/bilingual_cache')
RUNS_DIR = Path('/kaggle/working/bilingual_runs')
subprocess.run([
    'python','-m','ud_treebank_lab','make-bilingual-configs',
    '--manifest',str(manifest),'--source',str(hindi_source),
    '--config-dir',str(CONFIG_DIR),'--cache-root',str(CACHE_DIR),
    '--runs-root',str(RUNS_DIR),'--epochs','6','--batch-size','4'
], check=True)
configs = sorted(CONFIG_DIR.glob('*.yaml'))
print(f'{len(configs)} target-language configurations generated')


## Step 3 — Optional recovery from earlier Kaggle session

Before launching, restore the exact contents of `bilingual_cache/` and `bilingual_runs/` from a previously saved private Kaggle Dataset or rclone cloud backup to the corresponding `/kaggle/working/` directories. **Checkpoints and cache files are not inherently durable after the session ends.** Only use trusted checkpoints.


In [ ]:
# Optional manual restore example; edit paths to the PREVIOUS version output dataset:
# import shutil
# previous = Path('/kaggle/input/my-previous-version')
# shutil.copytree(previous/'bilingual_cache', CACHE_DIR, dirs_exist_ok=True)
# shutil.copytree(previous/'bilingual_runs', RUNS_DIR, dirs_exist_ok=True)
print('Restore step skipped; enable and edit if resuming a previous Kaggle session.')


## Step 4 — Inspect scheduler plan, then launch two workers

The launcher stages the shared frozen Hindi embeddings **once** in a separate process, then creates one target-language worker on GPU 0 and one on GPU 1. It queues the remaining languages and keeps separate logs/checkpoints. This cell can run a long time; Kaggle sessions may end before every language finishes.


In [ ]:
base = ['python','-m','ud_treebank_lab','launch-bilingual','--configs',*[str(c) for c in configs], '--gpus','0,1']
subprocess.run(base + ['--dry-run'], check=True)


In [ ]:
# Execute after verifying every path and your remaining GPU quota:
subprocess.run(base, check=True)


## Step 5 — Inspect logs and export target silver trees

Final treebank predictions require **both** Hindi source references and target sentences. This is not yet a target-only parser. Never interpret projection dev loss as true LAS.


In [ ]:
for cfg_path in configs:
    import yaml
    cfg = yaml.safe_load(cfg_path.read_text())
    run_dir = Path(cfg['output_dir'])
    print(cfg['language'], 'done:', (run_dir/'done.json').exists(),
          'checkpoint:', (run_dir/'last.pt').exists())
# Example for a finished Marathi run:
# subprocess.run(['udtl','predict-bilingual','--config',str(CONFIG_DIR/'hindi_to_mr_bilingual.yaml')],check=True)


## Step 6 — Back up BEFORE the session is lost

Save a notebook version or publish a private Kaggle Dataset with your runs/cache, or copy to Drive/Dropbox through `rclone` configured via Kaggle Secrets. Be mindful of the documented 20-GB saved working space. Keeping active and completed checkpoints is essential; model-only ZIPs are not sufficient for training resumption.

Example only, if `rclone` is already authorized:

```bash
rclone copy /kaggle/working/bilingual_runs drive:UDTreebankLab/runs -P
rclone copy /kaggle/working/bilingual_cache dropbox:UDTreebankLab/caches -P
```
